# Project 1 - Part 1


In [0]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.appName("my_project_1").getOrCreate()


Importing all spark data types and spark functions for your convenience.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

The type of household ID was originally a string, but we changed it to an integer to match its type in the reference data. This was done to enable a successful join.

In [0]:
# Read a CSV into a dataframe
# There is a smarter version, that will first check if there is a Parquet file and use it
def load_csv_file(filename, schema):
  # Reads the relevant file from distributed file system using the given schema

  allowed_files = {'Daily program data': ('Daily program data', "|"),
                   'demographic': ('demographic', "|")}

  if filename not in allowed_files.keys():
    print(f'You were trying to access unknown file \"{filename}\". Only valid options are {allowed_files.keys()}')
    return None

  filepath = allowed_files[filename][0]
  dataPath = f"dbfs:/mnt/coursedata2024/fwm-stb-data/{filepath}"
  delimiter = allowed_files[filename][1]

  df = spark.read.format("csv")\
    .option("header","false")\
    .option("delimiter",delimiter)\
    .schema(schema)\
    .load(dataPath)
  return df

# This dict holds the correct schemata for easily loading the CSVs
schemas_dict = {'Daily program data':
                  StructType([
                    StructField('prog_code', StringType()),
                    StructField('title', StringType()),
                    StructField('genre', StringType()),
                    StructField('air_date', StringType()),
                    StructField('air_time', StringType()),
                    StructField('Duration', FloatType())
                  ]),
                'viewing':
                  StructType([
                    StructField('device_id', StringType()),
                    StructField('event_date', StringType()),
                    StructField('event_time', IntegerType()),
                    StructField('mso_code', StringType()),
                    StructField('prog_code', StringType()),
                    StructField('station_num', StringType())
                  ]),
                'viewing_full':
                  StructType([
                    StructField('mso_code', StringType()),
                    StructField('device_id', StringType()),
                    StructField('event_date', IntegerType()),
                    StructField('event_time', IntegerType()),
                    StructField('station_num', StringType()),
                    StructField('prog_code', StringType())
                  ]),
                'demographic':
                  StructType([StructField('household_id',IntegerType()),
                    StructField('household_size',IntegerType()),
                    StructField('num_adults',IntegerType()),
                    StructField('num_generations',IntegerType()),
                    StructField('adult_range',StringType()),
                    StructField('marital_status',StringType()),
                    StructField('race_code',StringType()),
                    StructField('presence_children',StringType()),
                    StructField('num_children',IntegerType()),
                    StructField('age_children',StringType()), #format like range - 'bitwise'
                    StructField('age_range_children',StringType()),
                    StructField('dwelling_type',StringType()),
                    StructField('home_owner_status',StringType()),
                    StructField('length_residence',IntegerType()),
                    StructField('home_market_value',StringType()),
                    StructField('num_vehicles',IntegerType()),
                    StructField('vehicle_make',StringType()),
                    StructField('vehicle_model',StringType()),
                    StructField('vehicle_year',IntegerType()),
                    StructField('net_worth',IntegerType()),
                    StructField('income',StringType()),
                    StructField('gender_individual',StringType()),
                    StructField('age_individual',IntegerType()),
                    StructField('education_highest',StringType()),
                    StructField('occupation_highest',StringType()),
                    StructField('education_1',StringType()),
                    StructField('occupation_1',StringType()),
                    StructField('age_2',IntegerType()),
                    StructField('education_2',StringType()),
                    StructField('occupation_2',StringType()),
                    StructField('age_3',IntegerType()),
                    StructField('education_3',StringType()),
                    StructField('occupation_3',StringType()),
                    StructField('age_4',IntegerType()),
                    StructField('education_4',StringType()),
                    StructField('occupation_4',StringType()),
                    StructField('age_5',IntegerType()),
                    StructField('education_5',StringType()),
                    StructField('occupation_5',StringType()),
                    StructField('polit_party_regist',StringType()),
                    StructField('polit_party_input',StringType()),
                    StructField('household_clusters',StringType()),
                    StructField('insurance_groups',StringType()),
                    StructField('financial_groups',StringType()),
                    StructField('green_living',StringType())
                  ])
}

# Read demogrphic data


In [0]:
%%time
# demographic data filename is 'demographic'
demo_df = load_csv_file('demographic', schemas_dict['demographic'])
demo_df.count()
demo_df.printSchema()
print(f'demo_df contains {demo_df.count()} records!')
display(demo_df.limit(6))

root
 |-- household_id: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- num_adults: integer (nullable = true)
 |-- num_generations: integer (nullable = true)
 |-- adult_range: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- race_code: string (nullable = true)
 |-- presence_children: string (nullable = true)
 |-- num_children: integer (nullable = true)
 |-- age_children: string (nullable = true)
 |-- age_range_children: string (nullable = true)
 |-- dwelling_type: string (nullable = true)
 |-- home_owner_status: string (nullable = true)
 |-- length_residence: integer (nullable = true)
 |-- home_market_value: string (nullable = true)
 |-- num_vehicles: integer (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_model: string (nullable = true)
 |-- vehicle_year: integer (nullable = true)
 |-- net_worth: integer (nullable = true)
 |-- income: string (nullable = true)
 |-- gender_individual: string (nullable = 

household_id,household_size,num_adults,num_generations,adult_range,marital_status,race_code,presence_children,num_children,age_children,age_range_children,dwelling_type,home_owner_status,length_residence,home_market_value,num_vehicles,vehicle_make,vehicle_model,vehicle_year,net_worth,income,gender_individual,age_individual,education_highest,occupation_highest,education_1,occupation_1,age_2,education_2,occupation_2,age_3,education_3,occupation_3,age_4,education_4,occupation_4,age_5,education_5,occupation_5,polit_party_regist,polit_party_input,household_clusters,insurance_groups,financial_groups,green_living
15,2,2,1,000000000000100000000,S,B,null,null,0000000000000000000,000000000000000,S,O,5,E,null,null,null,null,6,4,M,60,4,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,D,443,02C3,08C3,null
24,2,2,1,000000000100000000000,null,W,null,null,0000000000000000000,000000000000000,M,O,null,F,null,null,null,null,7,7,F,46,3,Z,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,R,223,09O3,03O3,null
26,null,null,null,000000000000000000000,null,null,null,null,0000000000000000000,000000000000000,S,null,null,F,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,46G,04CG,08CG,null
28,3,2,2,000000110000000000000,S,W,Y,1,0000010000000000000,000001000000000,S,O,3,H,null,null,null,null,5,7,M,38,2,4,null,null,34,1,7,null,null,null,null,null,null,null,null,null,null,V,473,11R3,09C3,1
35,1,1,1,000000000100000000000,null,W,null,null,0000000000000000000,000000000000000,null,null,null,G,null,null,null,null,4,null,M,50,2,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,D,523,13C3,08C3,null
36,null,null,null,000000000000000000000,null,null,null,null,0000000000000000000,000000000000000,null,null,null,G,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,51G,10RG,10RG,null


CPU times: user 570 ms, sys: 15.1 ms, total: 585 ms
Wall time: 11.6 s


# Read Daily program data

In [0]:
%%time
# daily_program data filename is 'Daily program data'
daily_prog_df = load_csv_file('Daily program data', schemas_dict['Daily program data'])

daily_prog_df.printSchema()
print(f'daily_prog_df contains {daily_prog_df.count()} records!')
display(daily_prog_df.limit(6))

root
 |-- prog_code: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- air_date: string (nullable = true)
 |-- air_time: string (nullable = true)
 |-- Duration: float (nullable = true)

daily_prog_df contains 13194849 records!


prog_code,title,genre,air_date,air_time,Duration
EP000000250035,21 Jump Street,Crime drama,20151219,050000,60.0
EP000000250035,21 Jump Street,Crime drama,20151219,110000,60.0
EP000000250063,21 Jump Street,Crime drama,20151219,180000,60.0
EP000000510007,A Different World,Sitcom,20151219,100000,30.0
EP000000510008,A Different World,Sitcom,20151219,103000,30.0
EP000000510159,A Different World,Sitcom,20151219,080300,29.0


CPU times: user 980 ms, sys: 25.5 ms, total: 1.01 s
Wall time: 12.8 s


# Read viewing data

In [0]:
dataPath = "dbfs:/FileStore/ddm/10m_viewing"

viewing10m_df = spark.read.format("csv")\
    .option("header","true")\
    .option("delimiter",",")\
    .schema(schemas_dict['viewing_full'])\
    .load(dataPath)

display(viewing10m_df.limit(6))
print(f'viewing10m_df contains {viewing10m_df.count()} rows!')

mso_code,device_id,event_date,event_time,station_num,prog_code
01540,0000000050f3,20150222,193802,61812,EP009279780033
01540,0000000050f3,20150222,195314,31709,EP021056430002
01540,0000000050f3,20150222,200151,61812,EP009279780033
01540,000000005518,20150222,111139,46784,EP004891370013
01540,000000005518,20150222,190000,14771,EP012124070127
01540,000000005518,20150222,200000,14771,EP010237320166


viewing10m_df contains 9935852 rows!


# Read reference data

Note that we removed the 'System Type' column.

In [0]:
# Read the new parquet
ref_data_schema = StructType([
    StructField('device_id', StringType()),
    StructField('dma', StringType()),
    StructField('dma_code', StringType()),
    StructField('household_id', IntegerType()),
    StructField('zipcode', IntegerType())
])

# Reading as a Parquet
dataPath = f"dbfs:/ddm_course_staff_files/ref_data"
ref_data = spark.read.format('parquet') \
                    .option("inferSchema","true")\
                    .load(dataPath)
                    
display(ref_data.limit(600))
print(f'ref_data contains {ref_data.count()} rows!')

device_id,dma,dma_code,household_id,zipcode
0000000050f3,Toledo,547,1471346,43609
000000006785,Amarillo,634,1924512,79119
000000007320,Lake Charles,643,3154808,70634
000000007df9,Lake Charles,643,1924566,70601
000000009595,Lexington,541,1600886,40601
000000009c6a,Houston,618,1924713,77339
000000009daa,Oklahoma City,650,1924725,73703
000000009e5a,Houston,618,2935414,77345
00000000a215,Oklahoma City,650,3521041,73703
00000000a290,Little Rock-Pine Bluff,693,1924784,65616


ref_data contains 704172 rows!


In [0]:
# Filter columns that we will not use and remove duplicates
ref = ref_data.select("device_id", "household_id").distinct()
daily_prog = daily_prog_df.distinct()
viewing10m = viewing10m_df.select("device_id", "prog_code").distinct()
demo = demo_df.select("household_id", "vehicle_make", "num_adults", "age_individual", "age_2", "income").distinct()

Condition 1

In [0]:
# Condition 1
avg_duration = daily_prog.filter(col('duration').isNotNull()).select(avg('duration')).first()[0]  # Compute the average duration
# Add a column 'cond1' with value 1 if program duration is above average, 0 otherwise
daily_prog = daily_prog.withColumn("cond1", when(col("duration") > avg_duration, 1).otherwise(0))

Condition 6

In [0]:
# Condition 6
# Convert the genre string into an array to allow checking for specific genres
daily_prog = daily_prog.withColumn("genres_arr", split(col("genre"), ","))
# Add a column 'cond6' with value 1 if the program belongs to at least one of the required genres bellow, otherwise 0
daily_prog = daily_prog.withColumn("cond6", when(
    array_contains(col("genres_arr"), "Collectibles") |
    array_contains(col("genres_arr"), "Art") |
    array_contains(col("genres_arr"), "Snowmobile") |
    array_contains(col("genres_arr"), "Public affairs") |
    array_contains(col("genres_arr"), "Animated") |
    array_contains(col("genres_arr"), "Music"), 1).otherwise(0))
daily_prog = daily_prog.drop("genres_arr") # this column is not needed anymore

Condition 7

In [0]:
# Condition 7
# Add a column 'cond7' with value 1 if the program title contains at least two of the required words, otherwise 0
daily_prog = daily_prog.withColumn("cond7", when(
        (when(lower(col("title")).contains("better"), 1).otherwise(0) +
         when(lower(col("title")).contains("girls"), 1).otherwise(0) +
         when(lower(col("title")).contains("the"), 1).otherwise(0) +
         when(lower(col("title")).contains("call"), 1).otherwise(0)) >= 2, 1).otherwise(0)
)

Condition 4

In [0]:
# Condition 4 
# Prepare a helper DataFrame to compute whether the program started or ended on Friday the 13th
# Convert air_date and air_time into a proper timestamp for start_time
# Calculate end_time based on program duration (converted from minutes to seconds)
# Extract day of month and day of week for both start and end timestamps
# Define cond4 = 1 if either the start or end time falls on a Friday the 13th
cond4_helper = daily_prog.select("prog_code", "air_date", "air_time", "Duration") \
    .withColumn("air_time_str", format_string("%06d", col("air_time").cast("int"))) \
    .withColumn("air_datetime_str", concat_ws(" ", col("air_date"), col("air_time_str"))) \
    .withColumn("start_time", to_timestamp(col("air_datetime_str"), "yyyyMMdd HHmmss")) \
    .withColumn("duration_seconds", (col("Duration") * 60).cast("int")) \
    .withColumn("end_time", col("start_time") + col("duration_seconds").cast("interval second")) \
    .withColumn("start_date", dayofmonth(col("start_time"))) \
    .withColumn("start_day", dayofweek(col("start_time"))) \
    .withColumn("end_date", dayofmonth(col("end_time"))) \
    .withColumn("end_day", dayofweek(col("end_time"))) \
    .withColumn("cond4", when(
        ((col("start_date") == 13) & (col("start_day") == 6)) |
        ((col("end_date") == 13) & (col("end_day") == 6)), 1).otherwise(0))
    
# Group by prog_code and aggregate the maximum value of cond4 per program
# This ensures cond4=1 for a program if it was aired at least once on a Friday the 13th
cond4_by_prog = cond4_helper.groupBy("prog_code").agg(max("cond4").alias("cond4"))

# Join the aggregated cond4 values back to daily_prog to mark relevant programs
# Use LEFT JOIN to keep all programs and fill missing values with 0
daily_prog = daily_prog.join(cond4_by_prog, "prog_code", "left").fillna(0, ["cond4"]).distinct()

Condition 2

In [0]:
# Condition 2
# Join viewing data with reference data to get household_id for each device
viewing_with_household = viewing10m.join(ref, on="device_id", how="inner")

# Join with demographic data, keeping only households that own a Toyota (vehicle_make code "91")
toyota_viewing = viewing_with_household.join(
    demo.filter(col("vehicle_make") == "91"),
    on="household_id",
    how="inner"
)

# Get distinct prog_codes that were viewed by Toyota-owning households
toyota_prog_codes = toyota_viewing.select("prog_code").distinct()

# Add cond2 column to daily_prog and put 1 if the program was viewed by at least one Toyota household, 0 otherwise
# We use LEFT JOIN because we want to keep all programs in daily_prog even if they were not viewed by any Toyota household.
daily_prog = daily_prog.join(
    toyota_prog_codes.withColumn("cond2", lit(1)),
    on="prog_code",
    how="left"
).withColumn("cond2", when(col("cond2") == 1, 1).otherwise(0)).distinct()

Condition 3

In [0]:
# Condition 3
# Join viewing data with reference data to get household_id for each device (as we did in condition 2)
viewing_with_household = viewing10m.join(ref, on="device_id", how="inner")

# Filter demographic data to include only households that:
# - Have exactly 2 adults
# - Both age values are not null
# - The absolute age difference between the two adults is less than or equal to 6
demo_filtered = demo.filter(
    (col("num_adults") == 2) & col('age_individual').isNotNull() & col('age_2').isNotNull() &
    (abs(col("age_individual") - col("age_2")) <= 6)
)

# Join the filtered demographic households with viewing data to get relevant viewing records
matched_viewing = viewing_with_household.join(
    demo_filtered.select("household_id"),
    on="household_id",
    how="inner"
)

# Get distinct prog_codes that were viewed by households matching the conditions
prog_codes_condition3 = matched_viewing.select("prog_code").distinct()

# Add cond3 column to daily_prog and set it to 1 if the program was viewed by at least one matching household, 0 otherwise
# LEFT JOIN is used to keep all programs in daily_prog even if they were not viewed by such households
daily_prog = daily_prog.join(
    prog_codes_condition3.withColumn("cond3", lit(1)),
    on="prog_code",
    how="left"
).withColumn("cond3", when(col("cond3") == 1, 1).otherwise(0)).distinct()

Condition 5

In [0]:
# Condition 5

# Count the number of distinct devices per household using the reference data
device_count = ref.groupBy("household_id").agg(countDistinct("device_id").alias("device_count"))

# Calculate the average household income from the demographic data (excluding null values)
avg_income = demo.filter(col("income").isNotNull()).select(avg("income")).first()[0]

# Filter households that have more than 3 devices and income below the average income
filtered_households = device_count.join(
    demo.select("household_id", "income"),
    on="household_id",
    how="inner"
).filter((col('income').isNotNull()) & (col("device_count") > 3) & (col("income") < avg_income)).select("household_id").distinct()

# Join viewing data with reference data to get household_id for each device (same as condition 2 and 3)
viewing_with_household = viewing10m.join(ref, on="device_id", how="inner")

# Join with filtered households to keep only viewing records from households that fit the condition
matched_viewing = viewing_with_household.join(
    filtered_households,
    on="household_id",
    how="inner"
)

# Get distinct prog_codes viewed by these households
prog_codes_condition5 = matched_viewing.select("prog_code").distinct()

# Add cond5 column to daily_prog and set it to 1 if the program was viewed by at least one matching household, 0 otherwise
# LEFT JOIN is used to keep all programs in daily_prog even if they were not viewed by such households
daily_prog = daily_prog.join(
    prog_codes_condition5.withColumn("cond5", lit(1)),
    on="prog_code",
    how="left"
).withColumn("cond5", when(col("cond5") == 1, 1).otherwise(0)).distinct()

Final Calculation

In [0]:
# Sum all the conditions for each row
daily_prog = daily_prog.withColumn(
    "conditions_met",
    col("cond1") + col("cond2") + col("cond3") + col("cond4") + col("cond5") + col("cond6") + col("cond7")
)

# Sign the row as malicious if at least 4 conditions met. 1 - malicious, 0 - not malicious
daily_prog = daily_prog.withColumn("malicious", when(col("conditions_met") >= 4, 1).otherwise(0))

# For each title, calculate:
# - total number of program records
# - number of malicious records
# - percentage of malicious records
title_score = daily_prog.groupBy("title").agg(
    count("*").alias("total_records"),
    sum("malicious").alias("malicious_records")).withColumn("malicious_percentage", (col("malicious_records") / col("total_records")) * 100)

# Choosing the titles with malicious percentage above 40%
malicious_titles = title_score.filter(col("malicious_percentage") > 40)
malicious_titles = malicious_titles.select("title", "malicious_percentage")

# Display the top 20 titles with the highest malicious percentage
display(malicious_titles.orderBy(col("malicious_percentage").desc()).limit(20))

title,malicious_percentage
Crazy/Beautiful,100.0
"Carreras, Domingo, Pavarotti in Concert",100.0
Soundtrack Clips,100.0
Live From Holy Land,100.0
With Honors,100.0
Fox 26 Morning News Weekends 6am,100.0
News 12 Nightside,100.0
Jamie Marks Is Dead,100.0
Kitsap Scratch Bowlers League,100.0
Windy City Poker,100.0
